In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import wandb
import itertools
from datetime import datetime
from time import time
from tqdm import tqdm

from sklearn.neighbors import KDTree

from hydra import compose, initialize
from hydra.utils import instantiate
from omegaconf import OmegaConf

from torch.utils.data import DataLoader
from pytorch_metric_learning.distances import LpDistance

from opr.utils import set_seed
from opr.testing import get_recalls
from opr.trainers.place_recognition import UnimodalPlaceRecognitionTrainer

In [3]:
import math
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple, Union

import cv2
import open3d as o3d
import gdown
import numpy as np
import pandas as pd
import torch
from loguru import logger
from omegaconf import OmegaConf
from pandas import DataFrame
from torch import Tensor
from torch.utils.data import Dataset

from opr.datasets.augmentations import (
    DefaultCloudSetTransform,
    DefaultCloudTransform,
    DefaultImageTransform,
    DefaultSemanticTransform,
)
from opr.datasets.projection import Projector
from opr.datasets.soc_utils import (
    get_points_labels_by_mask,
    instance_masks_to_objects,
    pack_objects,
    semantic_mask_to_instances,
)
from opr.optional_deps import lazy

# Lazy-load MinkowskiEngine - will return real module or helpful stub
ME = lazy("MinkowskiEngine", feature="sparse convolutions")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


# Dataset

In [ ]:
class SberRoboticsDataset(Dataset):
    scans_path: Path 
    csv_path: Path 
    dataset_df: DataFrame 
    subset: Literal["train", "test"]
    pointcloud_set_transform: DefaultCloudSetTransform
    pointcloud_quantization_size: float
    timestamp_to_find = 1847.474614448

    def __init__(
        self,
        scans_path: Union[str, Path],
        csv_path: Union[str, Path],
        subset: Literal["train", "test"],
        positive_threshold: float = 10.0,
        negative_threshold: float = 50.0,
        pointcloud_quantization_size: float = 0.05
    ):
        self.scans_path = Path(scans_path)
        self.csv_path = Path(csv_path)
        self.subset = subset 
        self.pointcloud_quantization_size = pointcloud_quantization_size

        if not self.scans_path.exists():
            raise FileNotFoundError(f"Given scans_path={self.scans_path} doesn't exist")

        if not self.csv_path.exists():
            raise FileNotFoundError(f"Give csv_path={self.csv_path} doesn't exist")

        self.dataset_df = pd.read_csv(csv_path)
        self.dataset_df = self.dataset_df[self.dataset_df["subset"] == self.subset]
        self.dataset_df = self.dataset_df.reset_index(drop=True)
        if self.subset == "test":
            self.dataset_df["in_query"] = False
            split_index = self.dataset_df.loc[self.dataset_df['# ts'] == self.timestamp_to_find].index.to_numpy()[0]
            self.dataset_df.loc[np.arange(start=split_index + 5, stop=len(self.dataset_df)), "in_query"] = True 

        if positive_threshold < 0.0:
            raise ValueError(f"positive_threshold must be non-negative, but {positive_threshold!r} given.")
        if negative_threshold < 0.0:
            raise ValueError(f"negative_threshold must be non-negative, but {negative_threshold!r} given.")

        self.positives_index, self.nonnegative_index = self._build_indexes(
            positive_threshold, negative_threshold
        )
        self.positives_mask, self.negatives_mask = self._build_masks(positive_threshold, negative_threshold)

        self.pointcloud_set_transform = DefaultCloudSetTransform(
            train=(self.subset == "train")
        )

    def __len__(self) -> int:
        return len(self.dataset_df)

    def _build_indexes(
        self, positive_threshold: float, negative_threshold: float
    ) -> Tuple[List[Tensor], List[Tensor]]:
        """Build index of elements that satisfy a UTM distance threshold condition.

        Args:
            positive_threshold (float): The maximum UTM distance between two elements
                for them to be considered positive.
            negative_threshold (float): The maximum UTM distance between two elements
                for them to be considered non-negative.

        Returns:
            Tuple[List[Tensor], List[Tensor]]: Tuple (positive_indices, nonnegative_indices)
                of two lists of element indexes that satisfy the UTM distance threshold condition
                for each element in the dataset.
        """
        xyz = torch.tensor(
            self.dataset_df[["px", "py", "pz"]].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        distances = torch.cdist(xyz, xyz)

        positives_mask = (distances > 0) & (distances < positive_threshold)
        nonnegatives_mask = distances < negative_threshold

        # Convert the boolean masks to index tensors
        positive_indices = [torch.nonzero(row).squeeze(dim=-1) for row in positives_mask]
        nonnegative_indices = [torch.nonzero(row).squeeze(dim=-1) for row in nonnegatives_mask]

        return positive_indices, nonnegative_indices

    def _build_masks(self, positive_threshold: float, negative_threshold: float) -> Tuple[Tensor, Tensor]:
        """Build boolean masks for dataset elements that satisfy a UTM distance threshold condition.

        Args:
            positive_threshold (float): The maximum UTM distance between two elements
                for them to be considered positive.
            negative_threshold (float): The maximum UTM distance between two elements
                for them to be considered non-negative.

        Returns:
            Tuple[Tensor, Tensor]: A tuple of two boolean masks that satisfy the UTM distance threshold
                condition for each element in the dataset. The first mask contains the indices of elements
                that satisfy the positive threshold, while the second mask contains the indices of elements
                that satisfy the negative threshold.
        """
        xyz = torch.tensor(
            self.dataset_df[["px", "py", "pz"]].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        distances = torch.cdist(xyz, xyz)

        positives_mask = (distances > 0) & (distances < positive_threshold)
        negatives_mask = distances > negative_threshold

        return positives_mask, negatives_mask

    def _load_pc(self, idx: int, map_num: str) -> Tensor:
        lidar_ts = str(self.dataset_df["lidar_ts"].iloc[idx]).zfill(6)
        filepath = self.scans_path / map_num / "keyframe_map" / "scans" / f"{lidar_ts}.pcd"
        
        pc = o3d.io.read_point_cloud(filepath)
        pc = np.asarray(pc.points)
        pc = torch.from_numpy(pc).to(torch.float32)
        
        return pc

    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        row = self.dataset_df.iloc[idx]
        data = {"idx": torch.tensor(idx, dtype=torch.int)}
        data["pose"] = torch.tensor(
            row[["px", "py", "pz", "qx", "qy", "qz", "qw"]].to_numpy(dtype=np.float32)
        )
        map_num = row["map"]

        pc = self._load_pc(idx, map_num)
        data["pointcloud_lidar_coords"] = pc 
        data["pointcloud_lidar_feats"] = torch.ones_like(pc[:, :1])

        return data
    
    def _collate_data_dict(self, data_list: List[Dict[str, Tensor]]) -> Dict[str, Tensor]:
        result: Dict[str, Tensor] = {}
        result["idxs"] = torch.stack([e["idx"] for e in data_list], dim=0)
        for data_key in data_list[0].keys():
            if data_key == "idx":
                continue 
            elif data_key == "pose":
                result["poses"] = torch.stack([e["pose"] for e in data_list], dim=0)
            elif data_key == "pointcloud_lidar_coords":
                coords_list = [e["pointcloud_lidar_coords"] for e in data_list]
                feats_list = [e["pointcloud_lidar_feats"] for e in data_list]
                n_points = [int(e.shape[0]) for e in coords_list]
                coords_tensor = torch.cat(coords_list, dim=0).unsqueeze(0)
                coords_tensor = self.pointcloud_set_transform(coords_tensor)
                coords_list = torch.split(
                    coords_tensor.squeeze(0),
                    split_size_or_sections=n_points,
                    dim=0
                )

                quantized_coords_list = []
                quantized_feats_list = []
                for coords, feats in zip(coords_list, feats_list):
                    quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                        coordinates=coords,
                        features=feats,
                        quantization_size=self.pointcloud_quantization_size
                    )
                    quantized_coords_list.append(quantized_coords)
                    quantized_feats_list.append(quantized_feats)
                
                result["pointclouds_lidar_coords"] = ME.utils.batched_coordinates(quantized_coords_list)
                result["pointclouds_lidar_feats"] = torch.cat(quantized_feats_list)
            elif data_key == "pointcloud_lidar_feats":
                continue 
            else: 
                raise ValueError(f"Unknown data key: {data_key!r}")
        return result
    
    def collate_fn(self, data_list: List[Dict[str, Tensor]]) -> Dict[str, Tensor]:
        """Pack input data list into batch.

        Args:
            data_list (List[Dict[str, Tensor]]): batch data list generated by DataLoader.

        Returns:
            Dict[str, Tensor]: dictionary of batched data.
        """
        return self._collate_data_dict(data_list)

# Modify UnimodalTrainer

In [5]:
class ModifiedUnimodalPlaceRecognitionTrainer(UnimodalPlaceRecognitionTrainer):
    def _get_recalls(
        self,
        query_embs: np.ndarray,
        db_embs: np.ndarray,
        dist_matrix: np.ndarray,
        dist_thresh: float = 5.0,
        at_n: int = 5,
    ) -> Tuple[np.ndarray, float, Optional[float]]:
        """Calculate Recall@N, Recall@1% and mean top-1 distance for the given query and db embeddings.

        Args:
            query_embs (np.ndarray): Query embeddings array.
            db_embs (np.ndarray): Database embeddings array.
            dist_matrix (np.ndarray): Distance matrix of shape (query_len, db_len).
            dist_thresh (float): Distance threshold for positive match. Defaults to 5.0.
            at_n (int): The maximum N value for the Recall@N metric. Defaults to 5.

        Returns:
            Tuple[np.ndarray, float, Optional[float]]: (Recall@N, Recall@1%, mean top-1 distance).
                The 'mean top-1 distance' metric may be `None` if Recall@1 = 0.
        """
        database_tree = KDTree(db_embs)

        # Create positives mask based on distance threshold
        positives_mask = dist_matrix <= dist_thresh
        queries_with_matches = int((positives_mask.sum(axis=1) > 0).sum())
        
        if queries_with_matches == 0:
            logger.warning("No queries with matches found!")
            return np.zeros((at_n,), dtype=float), 0.0, None

        recall_at_n = np.zeros((at_n,), dtype=float)
        top1_distances = []
        one_percent_threshold = max(int(round(len(db_embs) / 100.0)), 1)

        k = min(at_n, len(db_embs))
        distances, indices = database_tree.query(query_embs, k=k)

        for query_i, closest_inds in enumerate(indices):
            query_gt_matches_mask = positives_mask[query_i][closest_inds]
            if query_gt_matches_mask[0]:
                top1_distances.append(distances[query_i][0])

            # Pad with False if we have fewer than at_n results
            if len(query_gt_matches_mask) < at_n:
                padded_mask = np.zeros(at_n, dtype=bool)
                padded_mask[:len(query_gt_matches_mask)] = query_gt_matches_mask
                query_gt_matches_mask = padded_mask
                
            recall_at_n += np.cumsum(query_gt_matches_mask, axis=0, dtype=bool)

        recall_at_n = recall_at_n / queries_with_matches
        # Ensure we don't try to access beyond the available recall values
        # For Recall@1%, use the minimum of one_percent_threshold and at_n
        recall_index = min(one_percent_threshold, at_n) - 1
        one_percent_recall = recall_at_n[recall_index]
        
        if len(top1_distances) > 0:
            mean_top1_distance = np.mean(top1_distances)
        else:
            mean_top1_distance = None

        return recall_at_n, one_percent_recall, mean_top1_distance

    def test(self, dataloader: DataLoader, distance_threshold: float = 5.0) -> None:
        """Evaluates the model on the test set.

        Args:
            dataloader (DataLoader): The data loader for the test set.
            distance_threshold (float): The distance threshold for a correct match. Defaults to 25.0.
        """
        logger.info("=> Test stage:")
        start_t = time()
        self.model.eval()
        with torch.no_grad():
            embeddings_list = []
            for batch in tqdm(dataloader, desc="Calculating test set descriptors", leave=False):
                batch = {e: batch[e].to(self.device) for e in batch}
                embeddings = self.model(batch)["final_descriptor"]
                embeddings_list.append(embeddings.cpu().numpy())
                torch.cuda.empty_cache()
            test_embeddings = np.vstack(embeddings_list)

        test_df = dataloader.dataset.dataset_df

        query_indices = test_df[test_df["in_query"]].index.tolist()
        database_indices = test_df[~test_df["in_query"]].index.tolist()

        logger.debug(f"Test embeddings: {test_embeddings.shape}")
        logger.debug(f"Number of query indices: {len(query_indices)}")
        logger.debug(f"Number of database indices: {len(database_indices)}")

        coords_cols = ["px", "py"]
        utms = torch.tensor(test_df[coords_cols].to_numpy())
        dist_fn = LpDistance(normalize_embeddings=False)
        dist_utms = dist_fn(utms).numpy()

        n = 5
        query_embs = test_embeddings[query_indices]
        database_embs = test_embeddings[database_indices]
    
        # Distance matrix between queries and database
        distances = dist_utms[query_indices][:, database_indices]

        recalls_at_n, recall_at_one_percent, mean_top1_distance = self._get_recalls(
            query_embs, database_embs, distances, at_n=n, dist_thresh=distance_threshold
        )

        elapsed_t = time() - start_t
        minutes, seconds = divmod(int(elapsed_t), 60)
        logger.info(f"Test time: {int(minutes):02d}:{int(seconds):02d}")
        logger.info(f"Mean Recall@N:\n{recalls_at_n}")
        logger.info(f"Mean Recall@1% = {recall_at_one_percent}")
        logger.info(f"Mean top-1 distance = {mean_top1_distance}")
        
        self._stats["test"] = {}
        self._stats["test"]["mean_recall_at_1"] = recalls_at_n[0]
        self._stats["test"]["mean_recall_at_3"] = recalls_at_n[2]
        self._stats["test"]["mean_recall_at_5"] = recalls_at_n[4]
        self._stats["test"]["mean_recall_at_1%"] = recall_at_one_percent
        self._stats["test"]["mean_top1_distance"] = mean_top1_distance

# Training

In [6]:
with initialize(version_base=None, config_path="../libs/OpenPlaceRecognition/configs/"):
    cfg = compose(config_name="train_unimodal_minkloc3d")

print(OmegaConf.to_yaml(cfg))

wandb:
  disabled: true
  project: SeqPlaceRecognition
debug: false
device: 0
seed: 3121999
num_workers: 4
exp_name: finetune_minkloc3d_with_sr_dataset
epochs: 80
batch_expansion_threshold: 0.7
checkpoints_dir: /home/docker_mmpr/multimodal-place-recognition/data/finetuned_checkpoints
dataset:
  scans_path: /home/docker_mmpr/Datasets/2025-03-26-mmpr-datasets/keyframe-lidar-maps/keyframe-lidar-maps/mmpr_dataset
  csv_path: /home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/full_dataset/dataset.csv
  positive_threshold: 3.0
  negative_threshold: 10.0
  pointcloud_quantization_size: 0.05
sampler:
  _target_: opr.samplers.BatchSampler
  batch_size: 16
  batch_size_limit: 128
  batch_expansion_rate: 1.4
  max_batches: null
  positives_per_group: 2
  seed: ${seed}
  drop_last: true
model:
  _target_: opr.models.place_recognition.MinkLoc3D
  in_channels: 1
  out_channels: 256
  num_top_down: 1
  conv0_kernel_size: 5
  block: BasicBlock
  layers:
  - 1
  - 1
  - 1
  pl

## Init dataloaders

In [7]:
train_dataset = SberRoboticsDataset(
    scans_path=cfg.dataset.scans_path,
    csv_path=cfg.dataset.csv_path,
    subset="train",
    positive_threshold=cfg.dataset.positive_threshold,
    negative_threshold=cfg.dataset.negative_threshold,
    pointcloud_quantization_size=cfg.dataset.pointcloud_quantization_size,
)
test_dataset = SberRoboticsDataset(
    scans_path=cfg.dataset.scans_path,
    csv_path=cfg.dataset.csv_path,
    subset="test",
    positive_threshold=cfg.dataset.positive_threshold,
    negative_threshold=cfg.dataset.negative_threshold,
    pointcloud_quantization_size=cfg.dataset.pointcloud_quantization_size,
)

train_sampler = instantiate(cfg.sampler, dataset=train_dataset)

dataloaders = {}

dataloaders["train"] = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=train_dataset.collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=True,
)
dataloaders["test"] = DataLoader(
    test_dataset,
    batch_size=1,
    collate_fn=test_dataset.collate_fn,
    num_workers=cfg.num_workers,
    pin_memory=True
)

In [8]:
len(dataloaders["train"].dataset.dataset_df), len(dataloaders["test"].dataset.dataset_df)

(6284, 3261)

## Init loss

In [9]:
loss_fn = instantiate(cfg.loss)
loss_fn

BatchHardTripletMarginLoss(
  (miner_fn): BatchHardTripletMiner(
    (distance): LpDistance()
  )
  (loss_fn): TripletMarginLoss(
    (distance): LpDistance()
    (reducer): AvgNonZeroReducer()
  )
)

## Init model

In [ ]:
model = instantiate(cfg.model)

# load pretrained NCLT checkpoint
ckpt = torch.load("/home/docker_mmpr/multimodal-place-recognition/data/checkpoints/minkloc3d_nclt.pth")
model.load_state_dict(ckpt)
# model = model.to(cfg.device)

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
2025-08-25 22:51:07.290 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


<All keys matched successfully>

## Init optimizer and scheduler

In [11]:
optimizer = instantiate(cfg.optimizer, params=model.parameters())
scheduler = instantiate(cfg.scheduler, optimizer=optimizer)

In [12]:
checkpoints_dir = Path(
    "/home/docker_mmpr/multimodal-place-recognition/data/finetuned_checkpoints/" + "v8"
)

In [13]:
(not cfg.debug and not cfg.wandb.disabled)

False

In [14]:
trainer = ModifiedUnimodalPlaceRecognitionTrainer(
    checkpoints_dir=checkpoints_dir,
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_expansion_threshold=cfg.batch_expansion_threshold,
    wandb_log=(not cfg.debug and not cfg.wandb.disabled),
    device=cfg.device,
)

In [15]:
trainer.train(epochs=cfg.epochs, train_dataloader=dataloaders["train"], test_dataloader=dataloaders["test"])

2025-08-25 22:51:07.589 | INFO     | opr.trainers.place_recognition.unimodal:train:113 - =====> Epoch:   1/80:
2025-08-25 22:51:07.589 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:251 - => Train stage:
Train:   0%|          | 0/394 [00:00<?, ?it/s]

2025-08-25 22:51:35.376 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:284 - Train time: 00:27
2025-08-25 22:51:35.376 | INFO     | opr.trainers.place_recognition.unimodal:_loop_epoch:285 - Train stats: {'loss': 0.24804809961827273, 'avg_embedding_norm': 8.430311822649186, 'num_triplets': 16.0, 'num_non_zero_triplets': 14.296954314720812, 'non_zero_rate': 0.8935596446700508, 'max_pos_pair_dist': 1.0874698278565091, 'max_neg_pair_dist': 1.1073819194650891, 'mean_pos_pair_dist': 0.7824574698348941, 'mean_neg_pair_dist': 0.8015308818841343, 'min_pos_pair_dist': 0.5548271579942122, 'min_neg_pair_dist': 0.649534362524294}
2025-08-25 22:51:35.377 | INFO     | __main__:test:73 - => Test stage:
2025-08-25 22:51:50.107 | DEBUG    | __main__:test:90 - Test embeddings: (3261, 256)
2025-08-25 22:51:50.108 | DEBUG    | __main__:test:91 - Number of query indices: 1626
2025-08-25 22:51:50.108 | DEBUG    | __main__:test:92 - Number of database indices: 1635
2025-08-25 22:51:50.660 | 

In [16]:
# best_ckpt = torch.load(str(checkpoints_dir / "best.pth"))
# trainer.model.load_state_dict(best_ckpt["model_state_dict"])

In [17]:
trainer.test(dataloaders["test"])

2025-08-25 23:39:55.844 | INFO     | __main__:test:73 - => Test stage:
Calculating test set descriptors:   0%|          | 0/3261 [00:00<?, ?it/s]

2025-08-25 23:40:09.734 | DEBUG    | __main__:test:90 - Test embeddings: (3261, 256)
2025-08-25 23:40:09.735 | DEBUG    | __main__:test:91 - Number of query indices: 1626
2025-08-25 23:40:09.735 | DEBUG    | __main__:test:92 - Number of database indices: 1635
2025-08-25 23:40:10.257 | INFO     | __main__:test:112 - Test time: 00:14
2025-08-25 23:40:10.258 | INFO     | __main__:test:113 - Mean Recall@N:
[0.89852399 0.92066421 0.93726937 0.94526445 0.95264453]
2025-08-25 23:40:10.258 | INFO     | __main__:test:114 - Mean Recall@1% = 0.9526445264452644
2025-08-25 23:40:10.258 | INFO     | __main__:test:115 - Mean top-1 distance = 0.9087148548248981
